<a href="https://colab.research.google.com/github/roxanapodean/Disertatie/blob/master/NN_Multiclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

In [ ]:
np.random.seed(0)
tf.random.set_seed(0)

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving mult25%EEC1_fuzzy100%TCO1_Inputs.asc to mult25%EEC1_fuzzy100%TCO1_Inputs.asc


In [ ]:
filename = "mult25%EEC1_fuzzy100%TCO1_Inputs.asc"

with open(filename, "r") as f:
    lines = f.readlines()

lines = [line.strip() for line in lines if line.strip() != ""]

rows = []
outputs = []
max_inputs = 0

for line in lines:
    values = [float(x.strip()) for x in line.split(",")]

    output = int(values[-1])
    input_row = values[:-1]

    rows.append(input_row)
    outputs.append(output)

    max_inputs = max(max_inputs, len(input_row))

inputs = np.zeros((len(rows), max_inputs))

for i, row in enumerate(rows):
    inputs[i, :len(row)] = row

outputs = np.array(outputs).astype(int)

print(inputs.shape)
print(outputs.shape)
print(np.unique(outputs, return_counts=True))

(122230, 6)
(122230,)
(array([0, 1, 2]), array([81449, 27153, 13628]))


In [ ]:
scaler = StandardScaler()
inputs = scaler.fit_transform(inputs)

In [ ]:
n = len(inputs)

train_end = int(0.30 * n)
val_end = int(0.40 * n)

x_train = inputs[:train_end]
y_train = outputs[:train_end]

x_val = inputs[train_end:val_end]
y_val = outputs[train_end:val_end]

x_test = inputs[val_end:]
y_test = outputs[val_end:]

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax")
])

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=6,
    batch_size=256,
    verbose=1
)

Epoch 1/6
144/144 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9026 - loss: 0.4369 - val_accuracy: 0.9647 - val_loss: 0.1329
Epoch 2/6
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9803 - loss: 0.0794 - val_accuracy: 0.9805 - val_loss: 0.0555
Epoch 3/6
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9885 - loss: 0.0421 - val_accuracy: 0.9868 - val_loss: 0.0355
Epoch 4/6
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9921 - loss: 0.0289 - val_accuracy: 0.9903 - val_loss: 0.0266
Epoch 5/6
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9937 - loss: 0.0221 - val_accuracy: 0.9913 - val_loss: 0.0219
Epoch 6/6
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9947 - loss: 0.0181 - val_accuracy: 0.9924 - val_loss: 0.0191


In [ ]:
scores = model.predict(x_test)

y_pred = np.argmax(scores, axis=1)

2292/2292 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step


In [ ]:
C = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])

print("Confusion matrix:")
print(C)

Confusion matrix:
[[48911     2    13]
 [   60 16252     0]
 [  492     7  7601]]


In [ ]:
overall_accuracy = np.trace(C) / np.sum(C) * 100

print(f"Overall Accuracy: {overall_accuracy:.2f}%")

Overall Accuracy: 99.22%


In [ ]:
class_names = ["Legitim", "Fuzzy", "Multiplication"]
num_classes = 3
total = np.sum(C)

for k in range(num_classes):
    TP = C[k, k]
    FP = np.sum(C[:, k]) - TP
    FN = np.sum(C[k, :]) - TP
    TN = total - TP - FP - FN

    TPR = TP / (TP + FN) * 100 if (TP + FN) != 0 else 0
    TNR = TN / (TN + FP) * 100 if (TN + FP) != 0 else 0
    FNR = FN / (TP + FN) * 100 if (TP + FN) != 0 else 0
    FPR = FP / (TN + FP) * 100 if (TN + FP) != 0 else 0

    accuracy = (TP + TN) / (TP + TN + FP + FN) * 100 if (TP + TN + FP + FN) != 0 else 0
    precision = TP / (TP + FP) * 100 if (TP + FP) != 0 else 0
    f1_score = (2 * TP) / (2 * TP + FP + FN) * 100 if (2 * TP + FP + FN) != 0 else 0

    print(f"\nClass {k} - {class_names[k]}")
    print(f"TP: {TP}")
    print(f"TN: {TN}")
    print(f"FP: {FP}")
    print(f"FN: {FN}")
    print(f"TPR: {TPR:.2f}")
    print(f"TNR: {TNR:.2f}")
    print(f"FPR: {FPR:.2f}")
    print(f"FNR: {FNR:.2f}")
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"Precision: {precision:.2f}%")
    print(f"F1Score: {f1_score:.2f}%")


Class 0 - Legitim
TP: 48911
TN: 23860
FP: 552
FN: 15
TPR: 99.97
TNR: 97.74
FPR: 2.26
FNR: 0.03
Accuracy: 99.23%
Precision: 98.88%
F1Score: 99.42%

Class 1 - Fuzzy
TP: 16252
TN: 57017
FP: 9
FN: 60
TPR: 99.63
TNR: 99.98
FPR: 0.02
FNR: 0.37
Accuracy: 99.91%
Precision: 99.94%
F1Score: 99.79%

Class 2 - Multiplication
TP: 7601
TN: 65225
FP: 13
FN: 499
TPR: 93.84
TNR: 99.98
FPR: 0.02
FNR: 6.16
Accuracy: 99.30%
Precision: 99.83%
F1Score: 96.74%
